## EDA 

In [3]:
import pandas as pd
import numpy as np

In [ ]:
movieDF = pd.read_csv("./data/movies.csv")
ratingsDF = pd.read_csv("./data/ratingsDF.csv")
castDF = pd.read_csv("./data/cast.csv")
crewDF = pd.read_csv("./data/crew.csv")
tagsDF = pd.read_csv("./data/tags.csv")

In [ ]:
merge_movie_and_cast = pd.merge(movieDF[['movieId']], castDF[['movie_id', 'person_id', 'character']], left_on='movieId', right_on='movie_id')
movies_with_cast = merge_movie_and_cast.drop_duplicates(subset=["movieId"], keep='first')
movies_with_cast.head()
print(f"Movies with cast: {movies_with_cast.shape[0]}\nMovies without cast:{movieDF.shape[0]-movies_with_cast.shape[0]}\nMovie with cast %: {(movies_with_cast.shape[0]/movieDF.shape[0]) * 100}")

merge_movie_and_crew = pd.merge(movieDF[['movieId']], crewDF[['movie_id', 'person_id', 'job']], left_on='movieId', right_on='movie_id')
movies_with_crew = merge_movie_and_crew.drop_duplicates(subset=["movieId"], keep='first')
movies_with_crew.head()
print(f"\n\nMovies with crew: {movies_with_crew.shape[0]}\nMovies without crew:{movieDF.shape[0]-movies_with_crew.shape[0]}\nMovie with crew %: {(movies_with_crew.shape[0]/movieDF.shape[0]) * 100}")

Movies with cast: 8997
Movies without cast:745
Movie with cast %: 92.35269965099569


Movies with crew: 9021
Movies without crew:721
Movie with crew %: 92.59905563539314


In [ ]:
user_counts = ratingsDF.groupby('userId').size()
movie_counts = ratingsDF.groupby('movieId').size()

for name, counts in [('users', user_counts), ('movies', movie_counts)]:
    print(f"--- {name} ---")
    print(counts.describe())  # gives mean, std, min, 25/50/75%, max
    print(f"< 5 ratingsDF: {(counts < 5).sum()} ({(counts < 5).mean()*100:.1f}%)")
    print(f"< 10 ratingsDF: {(counts < 10).sum()} ({(counts < 10).mean()*100:.1f}%)")

--- users ---
count     610.000000
mean      165.304918
std       269.480584
min        20.000000
25%        35.000000
50%        70.500000
75%       168.000000
max      2698.000000
dtype: float64
< 5 ratings: 0 (0.0%)
< 10 ratings: 0 (0.0%)
--- movies ---
count    9724.000000
mean       10.369807
std        22.401005
min         1.000000
25%         1.000000
50%         3.000000
75%         9.000000
max       329.000000
dtype: float64
< 5 ratings: 6074 (62.5%)
< 10 ratings: 7455 (76.7%)


In [ ]:
tag_counts_per_movie = tagsDF.groupby('movieId').size()
tag_counts_per_user = tagsDF.groupby('userId').size()

total_movies = movieDF['movieId'].nunique()
movies_with_tags = tagsDF['movieId'].nunique()

print(f"Movies with tags: {movies_with_tags} / {total_movies} ({movies_with_tags/total_movies*100:.1f}%)")
print(f"Unique tags (raw, uncleaned): {tagsDF['tag'].nunique()}")

print("\n--- tags per movie (movies that have >=1 tag) ---")
print(tag_counts_per_movie.describe())

print("\n--- tags per user ---")
print(tag_counts_per_user.describe())

Movies with tags: 1572 / 9742 (16.1%)
Unique tags (raw, uncleaned): 1589

--- tags per movie (movies that have >=1 tag) ---
count    1572.000000
mean        2.342875
std         5.562342
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max       181.000000
dtype: float64

--- tags per user ---
count      58.000000
mean       63.500000
std       215.118103
min         1.000000
25%         2.250000
50%         4.000000
75%        13.000000
max      1507.000000
dtype: float64


In [ ]:
dupes = ratingsDF.duplicated(subset=['userId', 'movieId']).sum()
print(f"Duplicate(user, movie) pairs: {dupes}")
assert ratingsDF['rating'].between(0.5, 5.0).all(), "Found ratings outside valid range"

#rating count per movie
rating_counts = ratingsDF.groupby('movieId').size().rename('n_ratings')

Duplicate(user, movie) pairs: 0
movieId
1         215
2         110
3          52
4           7
5          49
         ... 
193581      1
193583      1
193585      1
193587      1
193609      1
Name: n_ratings, Length: 9724, dtype: int64


In [25]:
ratings = pd.read_csv('./data/ratings.csv')

# sanity checks
dupes = ratings.duplicated(subset=['userId', 'movieId']).sum()
print(f"Duplicate (user, movie) pairs: {dupes}")
assert ratings['rating'].between(0.5, 5.0).all(), "Found ratings outside valid range"

# compute n (ratings count per movie) — feeds directly into effective_α
rating_counts = ratings.groupby('movieId').size().rename('n_ratings')

# ---------- 2. Movies + genres + tags ----------
movies = pd.read_csv('./data/movies.csv')
tags = pd.read_csv('./data/tags.csv')

# treat "(no genres listed)" as empty string, not a dropped row
movies['genres_clean'] = movies['genres'].replace('(no genres listed)', '').str.replace('|', ' ', regex=False)

# lowercase + strip tags before combining (avoids "Funny" vs "funny" duplication)
tags['tag_clean'] = tags['tag'].str.lower().str.strip()
tags_per_movie = tags.groupby('movieId')['tag_clean'].apply(lambda x: ' '.join(x)).rename('tags_combined')

movies = movies.merge(tags_per_movie, on='movieId', how='left')
movies['tags_combined'] = movies['tags_combined'].fillna('')

# final combined text field for TF-IDF
movies['content_text'] = (movies['genres_clean'] + ' ' + movies['tags_combined']).str.strip()

# attach rating count + confidence for soft weighting
movies = movies.merge(rating_counts, on='movieId', how='left')
movies['n_ratings'] = movies['n_ratings'].fillna(0).astype(int)
movies['effective_alpha'] = 0.7 * (movies['n_ratings'] / (movies['n_ratings'] + 5))

# ---------- 3. Cast/crew — ensure no nulls break similarity later ----------
cast = pd.read_csv('./data/cast.csv')
crew = pd.read_csv('./data/crew.csv')
# movies with no cast/crew simply won't appear in these tables — that's fine,
# just make sure your similarity code treats "missing" as empty list, not an error

print(movies[['movieId', 'title', 'content_text', 'n_ratings', 'effective_alpha']].head())

Duplicate (user, movie) pairs: 0
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        content_text  n_ratings  \
0  Adventure Animation Children Comedy Fantasy pi...        215   
1  Adventure Children Fantasy fantasy magic board...        110   
2                           Comedy Romance moldy old         52   
3                               Comedy Drama Romance          7   
4                            Comedy pregnancy remake         49   

   effective_alpha  
0         0.684091  
1         0.669565  
2         0.638596  
3         0.408333  
4         0.635185  


In [ ]:
# get the clean movies csv out
movies.to_csv('data/processed/movies_clean.csv', index=False)